# Training of a WGAN on disk images.

In [ ]:
import functools as ft
from collections.abc import Iterable
from typing import Any

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

## Defining abstract classes for generators and critics

In [ ]:
class GeneratorDisksNIROld(eqx.Module):
    """WGAN generator for monochromatic NIR disk images.

    **Attributes**

    - `layers`: Tuple containing the different neural network layers. The only
        requirement is that each layer should just be callable using the output of the
        preceding layer. The first layer needs to be able to accept a latent 'noise'
        vector of size `size_in`.
    - `size_in`: Size of the latent 'noise' input vector to the first layer.
    """

    layers: tuple[Any, ...]  # Different callable layers.
    size_in: int = eqx.field(static=True)  # Size of input latent vector.

    # TODO: implement
    def __init__(self, key: jax.Array) -> None:
        """**Arguments**

        - `key`: JAX PRNG key to initialise the model.
        """
        key, subkey1, subkey2, subkey3, subkey4, subkey5 = jax.random.split(key, 6)

        self.size_in = 100
        self.layers = (
            eqx.nn.Linear(
                100, 256 * 16 * 16, use_bias=True, key=subkey1
            ),  # Fully connected.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Lambda(
                lambda x: jnp.reshape(x, shape=(256, 16, 16))
            ),  # Reshape to 256 x 16 x 16.
            eqx.nn.ConvTranspose2d(
                in_channels=256,
                out_channels=128,
                kernel_size=(4, 4),
                stride=(2, 2),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey2,
            ),  # Transposed convolution to 128 x 32 x 32.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.ConvTranspose2d(
                in_channels=128,
                out_channels=64,
                kernel_size=(4, 4),
                stride=(2, 2),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey3,
            ),  # Transposed convolution to 64 x 64 x 64.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.ConvTranspose2d(
                in_channels=64,
                out_channels=32,
                kernel_size=(4, 4),
                stride=(2, 2),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey4,
            ),  # Transposed convolution to 32 x 128 x 128.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Conv2d(
                in_channels=32,
                out_channels=1,
                kernel_size=(5, 5),
                stride=(1, 1),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey5,
            ),  # Final regular convolution to 1 x 128 x 128.
            eqx.nn.Lambda(
                lambda x: jnp.tanh(x)
            ),  # Final activation to map to signed unit interval [-1, 1].
        )
        return

    def __call__(self, x: jax.Array) -> jax.Array:
        """Feed-forward pass through the neural network."""
        print(f"SHAPE OF INPUT X: {x.shape}")
        for layer in self.layers:
            print(f"CURRENTLY CONSIDERING LAYER OF TYPE '{type(layer)}'")
            x = layer(x)
            print(f"OUTPUT SHAPE: {x.shape}")
        return x


class GeneratorDisksNIR(eqx.Module):
    """WGAN generator for monochromatic NIR disk images with upsampling.

    **Attributes**

    - `layers`: Tuple containing the different neural network layers. The only
        requirement is that each layer should just be callable using the output of the
        preceding layer, producing a 3D image (Y, X, Channel indexing) cube from an
        input 1D latent vector of given `size_in`.
    - `size_in`: Size of the latent 'noise' input vector to the first layer.
    """

    layers: tuple[Any, ...]  # Different callable layers.
    size_in: int = eqx.field(static=True)  # Size of input latent vector.

    # TODO: implement
    def __init__(self, key: jax.Array) -> None:
        """**Arguments**

        - `key`: JAX PRNG key to initialise the model.
        """
        key, subkey1, subkey2, subkey3, subkey4, subkey5 = jax.random.split(key, 6)

        self.size_in = 100
        self.layers = (
            eqx.nn.Linear(
                self.size_in, 256 * 16 * 16, use_bias=True, key=subkey1
            ),  # Fully connected from shape (100,) to (256 * 16 * 16,).
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Lambda(
                lambda x: jnp.reshape(x, shape=(256, 16, 16))
            ),  # Reshape to 256 x 16 x 16.
            eqx.nn.Lambda(
                lambda x: jnp.repeat(
                    jnp.repeat(x, repeats=2, axis=1),
                    2,
                    axis=2,
                )
            ),  # Upsample to 256 x 32 x 32.
            eqx.nn.Conv2d(
                in_channels=256,
                out_channels=128,
                kernel_size=(3, 3),
                stride=(1, 1),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey2,
            ),  # Regular convolution to 128 x 32 x 32.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Lambda(
                lambda x: jnp.repeat(
                    jnp.repeat(x, repeats=2, axis=1),
                    2,
                    axis=2,
                )
            ),  # Upsample to 128 x 64 x 64.
            eqx.nn.Conv2d(
                in_channels=128,
                out_channels=64,
                kernel_size=(3, 3),
                stride=(1, 1),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey3,
            ),  # Regular convolution to 64 x 64 x 64.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Lambda(
                lambda x: jnp.repeat(
                    jnp.repeat(x, repeats=2, axis=1),
                    repeats=2,
                    axis=2,
                )
            ),  # Upsample to 64 x 128 x 128.
            eqx.nn.Conv2d(
                in_channels=64,
                out_channels=32,
                kernel_size=(3, 3),
                stride=(1, 1),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey4,
            ),  # Regular convolution to 32 x 128 x 128.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.1)
            ),  # Activation.
            eqx.nn.Conv2d(
                in_channels=32,
                out_channels=1,
                kernel_size=(5, 5),
                stride=(1, 1),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey5,
            ),  # Final regular convolution to 1 x 128 x 128.
            eqx.nn.Lambda(
                lambda x: jnp.tanh(x)
            ),  # Final activation to map to signed unit interval [-1, 1].
        )
        return

    def __call__(self, x: jax.Array) -> jax.Array:
        """Feed-forward pass through the neural network."""
        print("\n")
        print(f"SHAPE OF INPUT X: {x.shape}")
        for layer in self.layers:
            print(f"CURRENTLY CONSIDERING LAYER OF TYPE '{type(layer)}'")
            x = layer(x)
            print(f"OUTPUT SHAPE: {x.shape}")
        print("\n")
        return x


class CriticDisksNIR(eqx.Module):
    """WGAN critic for monochromatic NIR disk images.

    **Attributes**

    - `layers`: Tuple containing the different neural network layers. The only
        requirement is that each layer should just be callable using the output of the
        preceding layer. The first layer needs to be able to accept a 3D image (Y, X,
        Channel indexing) cube.
    """

    layers: tuple[Any, ...]  # Different callable layers.

    def __init__(self, key: jax.Array):
        """**Arguments**

        - `key`: JAX PRNG key to initialise the model.
        """
        key, subkey1, subkey2, subkey3, subkey4, subkey5 = jax.random.split(key, 6)

        self.layers = (
            eqx.nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=(5, 5),
                stride=(2, 2),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey1,
            ),  # Regular convolution to 32 x 64 x 64 (assuming input is 1 x 128 x 128).
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.25)
            ),  # Activation.
            eqx.nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=(3, 3),
                stride=(2, 2),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey2,
            ),  # Regular convolution to 64 x 32 x 32.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.25)
            ),  # Activation.
            eqx.nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=(3, 3),
                stride=(2, 2),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey3,
            ),  # Regular convolution to 128 x 16 x 16.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.25)
            ),  # Activation.
            eqx.nn.Conv2d(
                in_channels=128,
                out_channels=256,
                kernel_size=(3, 3),
                stride=(1, 1),
                padding="SAME",
                padding_mode="ZEROS",
                use_bias=True,
                key=subkey4,
            ),  # Regular convolution to 256 x 16 x 16.
            eqx.nn.Lambda(
                lambda x: jax.nn.leaky_relu(x, negative_slope=0.25)
            ),  # Activation.
            eqx.nn.Lambda(lambda x: jnp.ravel(x)),  # Ravel to shape (256 * 16 * 16,)
            eqx.nn.Linear(
                256 * 16 * 16, 1, use_bias=True, key=subkey5
            ),  # Final activation to scalar of shape (1,)
        )

        return

    def __call__(self, x: jax.Array) -> jax.Array:
        """Feed-forward pass through the neural network."""
        print("\n")
        print(f"SHAPE OF INPUT X: {x.shape}")
        for layer in self.layers:
            print(f"CURRENTLY CONSIDERING LAYER OF TYPE '{type(layer)}'")
            x = layer(x)
            print(f"OUTPUT SHAPE: {x.shape}")
        print("\n")
        return x

In [ ]:
gen_notrans = GeneratorDisksNIR(key=jax.random.key(43343))
gen_trans = GeneratorDisksNIROld(key=jax.random.key(43343))

key = jax.random.key(42)
key, subkey = jax.random.split(key, 2)
z_test = jax.random.normal(subkey, shape=(100,))
test_pass_notrans = gen_notrans(z_test)
test_pass_trans = gen_trans(z_test)

fig, ax = plt.subplots(1, 2, figsize=(10, 10))
ax[0].imshow(test_pass_notrans[0, :, :])
ax[0].set_title("Upsampling + Reg. Conv2d")
ax[1].imshow(test_pass_trans[0, :, :])
ax[1].set_title("Transposed Conv2d")
plt.savefig("/home/toond/Downloads/compare.png", dpi=200)
plt.show()

In [ ]:
key, subkey = jax.random.split(key, 2)
crit = CriticDisksNIR(key=key)

key, subkey = jax.random.split(key, 2)
img_test = jax.random.normal(subkey, shape=(1, 128, 128))

w_test = crit(img_test)

In [ ]:
@ft.partial(jax.jit, static_argnums=1)  # `static` must be a PyTree of non-arrays.
@jax.grad  # differentiates with respect to `params`, as it is the first argument
def get_grads(params, static, x, y):
    model = eqx.combine(params, static)
    pred_y = jax.vmap(model)(x)
    return jnp.mean((y - pred_y) ** 2)


key = jax.random.key(23232)
# key, subkey = jax.random.split(key, 2)
# x = jax.random.normal(subkey, shape=(5, 100))
# key, subkey = jax.random.split(key, 2)
# y = jax.random.normal(subkey, shape=(5, 100))

# gen_params, gen_static = eqx.partition(gen, filter_spec=eqx.is_array)
# get_grads(gen_params, gen_static, x, y)

In [ ]:
# key, subkey = jax.random.split(key, 2)
# x = jax.random.normal(subkey, shape=(100,))
# y = gen(x)

# x_batch = jax.random.normal(subkey, shape=(5, 100))
# y_batch = jax.vmap(gen)(x_batch)

In [ ]:
# print(y.shape, y_batch.shape)

In [ ]:
# print(y.shape)

In [ ]:
def train_wgan(
    gen: eqx.Module,
    crit: eqx.Module,
    opt_gen: optax.GradientTransformation,
    opt_crit: optax.GradientTransformation,
    training_loader: Iterable,
    *,
    ngen: int,
    ncrit_ratio: int,
    key: jax.Array,
    size_in: int = 0,
    diagnostics: bool = False,
) -> None:
    """Trains a WGAN consisting of a generator and a critic using the Wasserstein
    metric under the Kantorovich-Rubinstein duality (minimisation of the critic).

    **Arguments**

    - `gen`: The WGAN generator. This should produce either a 3D image (Y, X, Channel
        indexing) cube from an input 1D latent vector.
    - `crit`: The WGAN critic. This should produce a scalar from a 3D image (Y, X,
        Channel indexing) cube.
    - `opt_gen`: Optax optimizer for the generator.
    - `opt_crit`: Optax optimizer for the critic.
    - `training_loader`: Image loader returning batches of the training data in 4D
        arrays using (Batch, Y, X, Channel) index ordering.
    - `ngen`: Number of generator training steps to take.
    - `ncrit_ratio`: Ratio of critic training steps per generator training steps
        (should idea be `>= 5` to ensure the critic remains close to optimality).
    - `key`: JAX PRNG key used for e.g. generator latent random input vector
        generation.
    - `size_in`: Generator input vector size. This does not need to be specified if
        the passed along generator `gen` already has a `size_in` attribute, which
        takes priority. In case of the latter, be sure to mark this attribute as static
        with `eqx.field(static=True)`.eqx.field(static=True)
    - `diagnostics`: Whether to create extra diagnostic outputs (e.g. plots) during
        training.
    """
    # Check if `gen` input size has been given either in `gen` or in function arguments.
    if hasattr(gen, "size_in"):
        size_in = getattr(gen, "size_in")
        print(
            f"Latent generator input size derived from 'size_in' attribute: {size_in}"
        )
    elif not hasattr(gen, "size_in") and (size_in == 0):
        raise ValueError(
            "Argument 'size_in' is `0` and generator has no 'size_in' attribute."
            "Cannot infer the size of the latent vector space."
            "Please specify either (the generator attribute having priority)."
        )
    else:
        size_in = size_in
        print(f"Latent generator input size derived from 'size_in' argument: {size_in}")

    # Check wheter output of generator is compatible with critic input,
    # and if the dimensionality of everything makes sense.
    key, subkey = jax.random.split(key, 2)
    _wgan_check_gen_and_crit(gen, crit, size_in=size_in, key=subkey)

    # TODO: check if the format of the training images actually matches the generator
    # output and critic input.
    training_img_iter = iter(training_loader)  # Training image iterator.
    print(type(training_img_iter))

    pass


def _wgan_check_gen_and_crit(
    gen: eqx.Module, crit: eqx.Module, *, size_in: int, key: jax.Array
) -> None:
    """Check the shape compatibility of the generator and critic.

    **Arguments**

    - `gen`: The WGAN generator. This should produce either a 3D image (Y, X, Channel
        indexing) cube from an input 1D latent vector.
    - `crit`: The WGAN critic. This should produce a scalar from a 3D image (Y, X,
        Channel indexing) cube.
    - `size_in`: Size of 1D generator input latent vector.
    - `key`: JAX PRNG key used to generate a generator latent random input vector for
        testing.
    """
    key, subkey = jax.random.split(key, 2)
    z_test = jax.random.normal(subkey, shape=(size_in,))
    img_gen_test = gen(z_test)
    if img_gen_test.ndim != 3:
        raise ValueError(
            "Generator should produce 3D image cubes using "
            "(Channel, Y, X) indexing. Instead found a generator"
            f"output with dimensionality {img_gen_test.ndim}!"
        )
    print(
        "Shape of generator output image cube using (Channel, Y, X) indexing: "
        f"{img_gen_test.shape}"
    )
    w_crit_test = crit(img_gen_test)
    if w_crit_test != (1,):
        raise ValueError(
            "The shape of the WGAN critic output should be (1,), i.e. a scalar."
            f"Instead found shape {w_crit_test.shape}. This is likely due to a"
            f"mismatch between the generator output shape {img_gen_test.shape}"
            "and the critic architecture."
        )

    return


# TODO: implement.
def _wgan_check_gen_and_training_loader():
    """Check the shape compatibility of the generator and critic.

    **Arguments**

    - `gen`: The WGAN generator. This should produce either a 3D image (Y, X, Channel
        indexing) cube from an input 1D latent vector.
    - `crit`: The WGAN critic. This should produce a scalar from a 3D image (Y, X,
        Channel indexing) cube.
    - `size_in`: Size of 1D generator input latent vector.
    - `key`: JAX PRNG key used to generate a generator latent random input vector for
        testing.
    """
    pass